# NeMo Forced Aligner Pipeline

This notebook has been refactored into a configurable pipeline. 
All you need to do is fill out the configuration cell below and then run all the cells.

In [ ]:
# ===============================================================================
#                            CONFIGURATION
# ===============================================================================

# --- Core Settings ---
LANGUAGE = "en"  # "en" or "de"
INPUT_TYPE = "spoken" # "vocals" or "spoken"
USE_PREDICTED_TEXT = False # Set to True if you don't have a transcript

# --- Alignment Settings ---                             
# "word": For word-by-word highlighting.        
# "word_separated": For word-by-word highlighting with an added invisible separator character.                                  
# "segment_manual": For phrase-by-phrase highlighting. Use '|' in RAW_TEXT to separate phrases.                                                       │     
# "segment_auto": Let the aligner automatically create segments based on punctuation.                                                                     │     
ALIGNMENT_TYPE = "segment_auto"

# --- Video Settings ---
# Set to a color name (e.g., "black", "blue") or "image" to use an image file from your WORK_DIR.
VIDEO_BACKGROUND = "#7e727bff"
VIDEO_RESOLUTION = "1280x720"


# --- File Paths ---
NEMO_DIR_PATH = "/workspace/NeMo"
WORK_DIR = "/workspace/nfa_tutorial/WORK_DIR"

# --- Model Selection ---
# This will be automatically selected based on LANGUAGE and INPUT_TYPE
# but you can override it here if you want.
PRETRAINED_MODEL = "stt_en_conformer_ctc_xlarge" # e.g., "stt_en_fastconformer_hybrid_large_pc"



# --- Subtitle Styling ---
ASS_FONTSIZE = "30"
VERTICAL_ALIGNMENT = "bottom"
TEXT_ALREADY_SPOKEN_RGB = "#5A5A5A"
TEXT_BEING_SPOKEN_RGB = "#FFABC4"
TEXT_NOT_YET_SPOKEN_RGB = "#FFFFFF"

# --- Input Text ---
# Paste your text here. It will be automatically formatted based on
# the ALIGNMENT_LEVEL setting. 
# Copy this field as the invisible seperator: "⠀" is the Braille Pattern Blank character (Unicode U+2800). 
# It's a non-printing character that appears as a space. 
RAW_TEXT = """ 
How to Fail Spectacularly at Life Without really trying. 
Welcome to my stream, broadcasting live from the couch 
I haven't moved off in years. 
Today, I'm sharing my tried and tested strategies for ensuring 
you don't - let's repeat that - You do not succeed at anything! 
Let's jump in.
Step one Avoid making decisions... at all costs!
Why commit to a path when you can endlessly scroll through life 
like it's a Netflix menu? Decisions are hard, so: just don't make them!
You can't make the wrong choice if you never make one.
"""

# Pipeline Steps

In [ ]:
import os
import json
import subprocess
import string
import glob
from PIL import Image
import shutil
from IPython.display import HTML, display, Code
from base64 import b64encode

def setup_environment(work_dir, nemo_dir):
    """Sets up the environment for the pipeline."""
    os.makedirs(work_dir, exist_ok=True)
    print(f"Project files will be stored in: {work_dir}")
    
    if not os.path.exists(nemo_dir):
        raise FileNotFoundError(f"Could not find NeMo directory at {nemo_dir}")
    print(f"NeMo toolkit found at: {nemo_dir}")
    

def hex_to_rgb(color):
    if isinstance(color, list):
        return color  # It's already an RGB list
    if isinstance(color, str):
        hex_color = color.lstrip('#')
        return list(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return color # Return as is if it's not a list or a string


def select_model(language, input_type, override_model):
    """Selects the pretrained model based on language and input type."""
    if override_model:
        print(f"Using override model: {override_model}")
        return override_model
    
    model_map = {
        "en": {
            "spoken": "stt_en_fastconformer_hybrid_large_pc",
            "vocals": "stt_en_fastconformer_hybrid_large_pc", # Replace with a suitable model for vocals if available
        },
        "de": {
            "spoken": "stt_de_fastconformer_hybrid_large_pc",
            "vocals": "stt_de_fastconformer_hybrid_large_pc", # Replace with a suitable model for vocals if available
        }
    }
    
    try:
        model = model_map[language][input_type]
        print(f"Selected model: {model}")
        return model
    except KeyError:
        raise ValueError(f"No model found for language='{language}' and input_type='{input_type}'. You can set a PRETRAINED_MODEL manually in the configuration.")

def prepare_media_files(work_dir, video_background, video_resolution):
    print("Starting media preparation...")
    video_extensions = ['mp4', 'mov', 'avi', 'mkv']
    audio_extensions = ['mp3', 'aac', 'm4a', 'flac', 'wma', 'wav']
    image_extensions = ['jpg', 'jpeg', 'png']

    # Find media files (case-insensitive)
    video_files = []
    for ext in video_extensions:
        video_files.extend(glob.glob(os.path.join(work_dir, f'*.{ext.lower()}')))
        video_files.extend(glob.glob(os.path.join(work_dir, f'*.{ext.upper()}')))

    audio_files = []
    for ext in audio_extensions:
        audio_files.extend(glob.glob(os.path.join(work_dir, f'*.{ext.lower()}')))
        audio_files.extend(glob.glob(os.path.join(work_dir, f'*.{ext.upper()}')))

    image_files = []
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(work_dir, f'*.{ext.lower()}')))
        image_files.extend(glob.glob(os.path.join(work_dir, f'*.{ext.upper()}')))

    print(f"Found video files: {video_files}")
    print(f"Found audio files: {audio_files}")
    print(f"Found image files: {image_files}")

    if video_files:
        input_video_path = video_files[0]
        output_basename = os.path.splitext(os.path.basename(input_video_path))[0]
        mono_wav_path = os.path.join(work_dir, f"{output_basename}_temp16.wav")
        output_video_path = os.path.join(work_dir, f"{output_basename}.mp4")

        print(f"Found video file: {input_video_path}")

        if not os.path.exists(output_video_path):
            shutil.copy(input_video_path, output_video_path)
            print(f"Copied video to {output_video_path}")

        if not os.path.exists(mono_wav_path):
            print(f"Extracting audio from '{input_video_path}' to 16-bit mono WAV...")
            subprocess.run(["ffmpeg", "-i", input_video_path, "-acodec", "pcm_s16le", "-ac", "1",
                            mono_wav_path], 
                        check=True, capture_output=True, text=True)
            print(f"Successfully created 16-bit mono WAV file: '{mono_wav_path}'")
        else:
            print(f"Mono WAV file already exists: '{mono_wav_path}'")
        return output_video_path, output_basename, mono_wav_path

    if audio_files:
        input_audio_path = audio_files[0]
        output_basename = os.path.splitext(os.path.basename(input_audio_path))[0]
        mono_wav_path = os.path.join(work_dir, f"{output_basename}_temp16.wav")
        output_video_path = os.path.join(work_dir, f"{output_basename}.mp4")

        print(f"Found audio file: {input_audio_path}")

        if not os.path.exists(mono_wav_path):
            print(f"Converting '{input_audio_path}' to 16-bit mono WAV...")
            subprocess.run(["ffmpeg", "-i", input_audio_path, "-acodec",
                            "pcm_s16le", "-ac", "1",
                            mono_wav_path], 
                        check=True, capture_output=True, text=True)
            print(f"Successfully created 16-bit mono WAV file: '{mono_wav_path}'")
        else:
            print(f"Mono WAV file already exists: '{mono_wav_path}'")

        if not os.path.exists(output_video_path):
            if video_background == 'image' and image_files:
                image_path = image_files[0]
                print(f"Creating video from image '{image_path}' and audio '{input_audio_path}'...")
                subprocess.run(["ffmpeg", "-loop", "1", "-framerate", "30", 
                                "-i", image_path, "-i",
                                input_audio_path, "-c:v", "libx264", "-preset", "p1", "-tune", "hq", "-cq", "19", "-c:a", "libmp3lame",
                                "-b:a", "192k", "-vf", 
                                f"scale={video_resolution}:force_original_aspect_ratio=decrease,pad={video_resolution}:-1:-1:color=black", 
                                "-shortest", "-y", output_video_path], check=True, capture_output=True, text=True)
                print(f"Successfully created video: '{output_video_path}'")
            else:
                ffmpeg_color = f"0x{video_background.lstrip('#')}"
                print(f"Creating video with '{video_background}' background from audio file '{input_audio_path}'...")
                subprocess.run(["ffmpeg", "-f", "lavfi", "-i",
                                f"color=c={ffmpeg_color}:s={video_resolution}", "-i", input_audio_path, "-c:v", "libx264", "-c:a", "copy", "-shortest", output_video_path, "-y"], check=True, capture_output=True, text=True)
                print(f"Successfully created video: '{output_video_path}'")
        else:
            print(f"Video file already exists: '{output_video_path}'")
        return output_video_path, output_basename, mono_wav_path

    raise FileNotFoundError(f"No audio or video files found in {work_dir}")

def prepare_text(raw_text, alignment_type):
    separator = "⠀ "
    
    if alignment_type == "word":
        punc_to_remove = string.punctuation.replace('!', '').replace('?', '').replace('-', '')
        processed_text = raw_text.translate(str.maketrans('', '', punc_to_remove))
        words = processed_text.split()
        return " ".join(words)

    elif alignment_type == "word_separated":
        words = raw_text.translate(str.maketrans('', '', string.punctuation)).split()
        return separator.join(words)
        
    elif alignment_type == "segment_manual":
        processed_text = raw_text.replace('|', separator)
        parts = processed_text.split(separator)
        cleaned_parts = [" ".join(part.split()) for part in parts]
        return separator.join(cleaned_parts)

    elif alignment_type == "segment_auto":
        return raw_text.replace('|', '').replace('⠀', '')
        
    else:
        raise ValueError(f"Unknown alignment_type: {alignment_type}")

def create_manifest(work_dir, text, mono_wav_path, output_basename):
    manifest_filepath = os.path.join(work_dir, f"{output_basename}_manifest.json")
    manifest_data = {
        "audio_filepath": mono_wav_path,
        "text": text
    }
    with open(manifest_filepath, 'w', encoding="utf-8") as f:
        line = json.dumps(manifest_data)
        f.write(line + "\n")
    print(f"Manifest file created at: {manifest_filepath}")
    return manifest_filepath

def run_forced_alignment(work_dir, nemo_dir, pretrained_model, use_pred_text, alignment_type,
vertical_alignment, ass_fontsize, text_already_spoken_rgb, text_being_spoken_rgb, text_not_yet_spoken_rgb, manifest_filepath):
    output_dir = os.path.join(work_dir, "nfa_output")

    # Convert hex colors to RGB lists for NeMo
    text_already_spoken_rgb_list = hex_to_rgb(text_already_spoken_rgb)
    text_being_spoken_rgb_list = hex_to_rgb(text_being_spoken_rgb)
    text_not_yet_spoken_rgb_list = hex_to_rgb(text_not_yet_spoken_rgb)

    command = ["python", f"{nemo_dir}/tools/nemo_forced_aligner/align.py",
                f"pretrained_name={pretrained_model}",
                f"manifest_filepath={manifest_filepath}",
                f"output_dir={output_dir}",
                f"align_using_pred_text={use_pred_text}",
                f"ass_file_config.fontsize={ass_fontsize}", 
                f"ass_file_config.vertical_alignment={vertical_alignment}",
                f"ass_file_config.text_already_spoken_rgb={text_already_spoken_rgb_list}",
                f"ass_file_config.text_being_spoken_rgb={text_being_spoken_rgb_list}",
                f"ass_file_config.text_not_yet_spoken_rgb={text_not_yet_spoken_rgb_list}"]

    if alignment_type == "word":
        command.append(f"ass_file_config.resegment_text_word_by_word=true")
    elif alignment_type in ["word_separated", "segment_manual"]:
        command.append(f"additional_segment_grouping_separator=['⠀']")
    elif alignment_type == "segment_auto":
        command.append(f"additional_segment_grouping_separator=['.', ',', ':', '?', '!', '...']")


    print("Running NeMo Forced Aligner...")
    try:
        subprocess.run(command, check=True, capture_output=True, text=True)
        print("NeMo Forced Aligner finished successfully.")
    except subprocess.CalledProcessError as e:
        print("NeMo Forced Aligner failed.")
        print("Return code:", e.returncode)
        print("Stderr:", e.stderr)
        raise e

def rename_alignment_outputs(work_dir, output_basename, mono_wav_path):
    """Renames the output files from the NeMo Forced Aligner to match the original media name."""
    print("Renaming alignment output files...")
    nfa_output_dir = os.path.join(work_dir, "nfa_output")
    original_stem = os.path.splitext(os.path.basename(mono_wav_path))[0]

    for dirpath, _, filenames in os.walk(nfa_output_dir):
        for filename in filenames:
            if filename.startswith(original_stem):
                old_path = os.path.join(dirpath, filename)
                new_filename = filename.replace(original_stem, output_basename)
                new_path = os.path.join(dirpath, new_filename)
                os.rename(old_path, new_path)
                print(f"Renamed '{old_path}' to '{new_path}'")
    
def create_srt_file(work_dir, mp4_path):
    input_basename = os.path.splitext(os.path.basename(mp4_path))[0]
    ass_word_path = os.path.join(work_dir, "nfa_output", "ass", "words", f"{input_basename}.ass")
    srt_path = os.path.join(work_dir, f"{input_basename}.srt")

    if not os.path.exists(ass_word_path):
        print("No word-level ASS file found to create SRT from. Skipping this step.")
        return

    print("Creating SRT file...")
    subprocess.run(["ffmpeg", "-y", "-i", ass_word_path, "-c:s", "text", srt_path], check=True, capture_output=True, text=True)
    print(f"SRT file created at: {srt_path}")
      
def generate_subtitled_video(work_dir, mp4_path):
    input_basename = os.path.splitext(os.path.basename(mp4_path))[0]

    ass_token_path = os.path.join(work_dir, "nfa_output", "ass", "tokens", f"{input_basename}.ass")
    ass_word_path = os.path.join(work_dir, "nfa_output", "ass", "words", f"{input_basename}.ass")

    if not os.path.exists(mp4_path):
        print("No video file found to add subtitles to. Skipping this step.")
        return

    # Word-level video
    if os.path.exists(ass_word_path):
        output_video_word = os.path.join(work_dir, f"{input_basename}_word_subtitles.mkv")
        print("Generating word-level subtitled video (MKV)...")
        subprocess.run(["ffmpeg", "-y", "-i", mp4_path, "-i", ass_word_path, "-c", "copy", "-map", "0", "-map", "1", output_video_word], check=True, capture_output=True, text=True)
        print(f"Word-level subtitled video created at: {output_video_word}")

    # Token-level video
    if os.path.exists(ass_token_path):
        output_video_token = os.path.join(work_dir, f"{input_basename}_token_subtitles.mkv")
        print("Generating token-level subtitled video (MKV)...")
        subprocess.run(["ffmpeg", "-y", "-i", mp4_path, "-i", ass_token_path, "-c", "copy", "-map", "0", "-map", "1", output_video_token], check=True, capture_output=True, text=True)
        print(f"Token-level subtitled video created at: {output_video_token}")

def move_files_to_done_directory(work_dir, output_basename):
    """Moves all generated files for a run into a 'done' subdirectory."""
    print(f"Moving files for '{output_basename}' to done directory...")
    done_dir = os.path.join(work_dir, "done", output_basename)
    os.makedirs(done_dir, exist_ok=True)

    # These are files directly in the work_dir
    files_in_work_dir = glob.glob(os.path.join(work_dir, f"{output_basename}*"))

    for src_path in files_in_work_dir:
        if os.path.isfile(src_path):
            file_name = os.path.basename(src_path)
            dst_path = os.path.join(done_dir, file_name)
            shutil.move(src_path, dst_path)
            print(f"Moved '{src_path}' to '{dst_path}'")

    # Now handle the nfa_output directory
    nfa_output_dir = os.path.join(work_dir, "nfa_output")
    if os.path.exists(nfa_output_dir):
        for dirpath, _, filenames in os.walk(nfa_output_dir):
            for filename in filenames:
                if filename.startswith(output_basename):
                    src_path = os.path.join(dirpath, filename)
                    # Recreate the same subdirectory structure within the done_dir
                    relative_path = os.path.relpath(dirpath, work_dir)
                    dst_dir_for_file = os.path.join(done_dir, relative_path)
                    os.makedirs(dst_dir_for_file, exist_ok=True)
                    dst_path = os.path.join(dst_dir_for_file, filename)
                    shutil.move(src_path, dst_path)
                    print(f"Moved '{src_path}' to '{dst_path}'")

In [ ]:
# ===============================================================================
#                                   RUN
# ===============================================================================

# 1) Setup
setup_environment(WORK_DIR, NEMO_DIR_PATH)
PRETRAINED_MODEL = select_model(LANGUAGE, INPUT_TYPE, PRETRAINED_MODEL)

# 2) Prepare inputs
output_video_path, output_basename, mono_wav_path = prepare_media_files(WORK_DIR, VIDEO_BACKGROUND, VIDEO_RESOLUTION)
formatted_text = prepare_text(RAW_TEXT, ALIGNMENT_TYPE)
manifest_filepath = create_manifest(WORK_DIR, formatted_text, mono_wav_path, output_basename)

# 3) Run alignment
run_forced_alignment(
    WORK_DIR,
    NEMO_DIR_PATH,
    PRETRAINED_MODEL,
    USE_PREDICTED_TEXT,
    ALIGNMENT_TYPE,
    VERTICAL_ALIGNMENT,
    ASS_FONTSIZE,
    TEXT_ALREADY_SPOKEN_RGB,
    TEXT_BEING_SPOKEN_RGB,
    TEXT_NOT_YET_SPOKEN_RGB,
    manifest_filepath,
)

# 4) Rename and generate outputs
rename_alignment_outputs(WORK_DIR, output_basename, mono_wav_path)
create_srt_file(WORK_DIR, output_video_path)
generate_subtitled_video(WORK_DIR, output_video_path)

# 5) Move files to done directory
move_files_to_done_directory(WORK_DIR, output_basename)

print("✅ Pipeline finished successfully!")